# Introduction to Quantitative Biology — Lecture 1
## The simplest model of gene expression, and noisy divisions

*Luca Ciandrini — luca.ciandrini@umontpellier.fr*

We use this notebook **together in class**. You do not need to write any code.

**How to use it.** Before running a cell marked **🔮 Predict**, write down what you expect —
a number, a sketch, a yes/no. Then run it and compare. When the result surprises you, that is
the interesting part.

Contents:
1. Some numbers of a cell
2. The simplest model of gene expression: $dY/dt = \beta - \alpha Y$
3. Switching off production
4. Partitioning of molecules at cell division

Notation as in Alon, Chapter 1: $\beta$ production rate, $\alpha$ removal rate.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# colours used in all figures of the course notes
BLUE, CORAL, INK, GREY = "#008AAE", "#F06E76", "#231F20", "#8399A3"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

---
## 1. Some numbers of a cell

From Alon, Table 1.1 (*E. coli*): about $4\times10^6$ proteins per cell, 4500 genes, and one
protein per cell corresponds to a concentration of about 1 nM.

**🔮 Predict:** on average, how many copies of each type of protein does a cell contain?
And a protein at 5 nM?

In [ ]:
proteins_per_cell = 4e6
genes = 4500
print(f"copies per protein type, on average: {proteins_per_cell / genes:.0f}")

nM_per_copy = 1.0          # 1 copy per cell is about 1 nM (Alon, Table 1.1)
concentration = 5.0        # nM
print(f"a protein at {concentration} nM: about {concentration / nM_per_copy:.0f} copies per cell")

An average of almost a thousand copies, but many proteins — for example transcription factors —
are present at a few copies. Keep this in mind for Sections 3 and 4.

---
## 2. The simplest model of gene expression

$$\frac{dY}{dt} = \beta - \alpha Y, \qquad
Y(t) = \frac{\beta}{\alpha} + \Big(Y_0 - \frac{\beta}{\alpha}\Big)e^{-\alpha t}.$$

The solution was derived in class (two ways). Below it is written as a Python function.

In [ ]:
def Y_exact(t, beta, alpha, Y0=0.0):
    '''Solution of dY/dt = beta - alpha*Y with Y(0) = Y0.'''
    Y_st = beta / alpha
    return Y_st + (Y0 - Y_st) * np.exp(-alpha * t)

### 2a. Starting below or above the steady state

**🔮 Predict:** a gene starts with $Y_0 = 0$, another with $Y_0 = 2\,\beta/\alpha$ (twice the steady
state). Sketch both curves. What do they have in common?

In [ ]:
beta, alpha = 1.0, 1.0            # time is measured in units of 1/alpha
t = np.linspace(0, 5, 400)

plt.figure(figsize=(6, 3.5))
plt.plot(t, Y_exact(t, beta, alpha, Y0=0), color=BLUE, lw=2, label="$Y_0 = 0$")
plt.plot(t, Y_exact(t, beta, alpha, Y0=2), color=CORAL, lw=2, label=r"$Y_0 = 2\beta/\alpha$")
plt.axhline(beta / alpha, color=INK, ls="--", lw=1, label=r"steady state $\beta/\alpha$")
plt.xlabel(r"time $\alpha t$"); plt.ylabel(r"$Y$ (units of $\beta/\alpha$)")
plt.legend(); plt.show()

### 2b. The response time

Two genes have the same removal rate $\alpha$, but the second is produced **ten times faster**.

**🔮 Predict:** which one reaches **half of its own steady state** first?

In [ ]:
beta1, beta2, alpha = 1.0, 10.0, 1.0
t = np.linspace(0, 5, 400)
T_half = math.log(2) / alpha

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.8))
for b, col, lab in [(beta1, BLUE, r"$\beta_1$"), (beta2, CORAL, r"$\beta_2 = 10\,\beta_1$")]:
    Y = Y_exact(t, b, alpha)
    left.plot(t, Y, color=col, lw=2, label=lab)
    right.plot(t, Y / (b / alpha), color=col, lw=4 if b == beta1 else 1.6, label=lab)
left.set(xlabel=r"time $\alpha t$", ylabel=r"$Y$ (units of $\beta_1/\alpha$)", title="absolute level")
right.axhline(0.5, color=GREY, ls=":"); right.axvline(T_half, color=GREY, ls=":")
right.set(xlabel=r"time $\alpha t$", ylabel=r"$Y / Y_\mathrm{st}$", title="divided by the steady state")
left.legend(); right.legend(); plt.show()

print(f"T_1/2 = ln2 / alpha = {T_half:.3f} (in units of 1/alpha), for both genes")

The production rate $\beta$ changes **how much**; only $\alpha$ changes **how fast**.

**Try it:** change `alpha` to `2.0` in the cell above and run it again. What moves?

### 2c. Putting numbers on it: stable proteins

A stable protein is only diluted by growth: $\alpha = \ln 2/\tau$, with $\tau$ the generation time.

**🔮 Predict:** for an *E. coli* dividing every 30 min, what is $\alpha$, and what is the response
time? What if the protein is also degraded, with a degradation half-life of 20 min?

In [ ]:
tau = 30.0                                   # min, generation time
alpha_dil = math.log(2) / tau
print(f"stable protein:   alpha = {alpha_dil:.4f} /min,  T_1/2 = {math.log(2) / alpha_dil:.1f} min")

alpha_deg = math.log(2) / 20.0               # degradation half-life of 20 min
alpha_tot = alpha_dil + alpha_deg
print(f"degraded protein: alpha = {alpha_tot:.4f} /min,  T_1/2 = {math.log(2) / alpha_tot:.1f} min")
print(f"to keep the same steady state, beta must be {alpha_tot / alpha_dil:.2f} times larger")

---
## 3. Switching off production

A stable protein is at steady state. At $t = 0$ its gene is switched off: $\beta = 0$, so
$dY/dt = -\alpha Y$ and

$$Y(t) = Y_\mathrm{st}\, e^{-\alpha t}, \qquad \alpha = \frac{\ln 2}{\tau}.$$

**🔮 Predict:** a cell has 1000 copies of the protein. How many generations do we have to wait
until the protein has disappeared?

In [ ]:
tau = 30.0                                   # min, generation time
alpha = math.log(2) / tau                    # stable protein: dilution only
Y_st = 1000.0

t = np.linspace(0, 12 * tau, 600)            # 12 generations
Y = Y_st * np.exp(-alpha * t)

plt.figure(figsize=(6, 3.5))
plt.plot(t / tau, Y, color=BLUE, lw=2)
generations = np.arange(0, 13)
plt.plot(generations, Y_st * np.exp(-alpha * generations * tau), "o", color=BLUE)
plt.xlabel(r"generations $t/\tau$"); plt.ylabel("molecules per cell"); plt.show()

for n in [1, 2, 3, 10, 12]:
    print(f"after {n:2d} generations: {Y_st * np.exp(-alpha * n * tau):8.3f} molecules per cell")

$Y$ is halved at every generation, $Y(n\tau) = Y_\mathrm{st}/2^n$. It goes to zero, but it never
becomes *exactly* zero.

Molecules, however, come one by one.

**🔮 Predict:** after how many generations is there **less than one molecule** per cell, for a
protein that starts with 10, 100 or 1000 copies? (A logarithmic axis turns the halving into a
straight line.)

In [ ]:
t = np.linspace(0, 12 * tau, 600)
plt.figure(figsize=(6, 3.8))
for Y_st, col in [(10, CORAL), (100, GREY), (1000, BLUE)]:
    plt.semilogy(t / tau, Y_st * np.exp(-alpha * t), color=col, lw=2, label=f"$Y_{{st}}$ = {Y_st}")
    n_gone = math.log2(Y_st)                 # Y_st / 2**n = 1
    plt.plot([n_gone], [1], "o", color=col, ms=8)
    print(f"Y_st = {Y_st:4d}: less than one molecule after {n_gone:.1f} generations "
          f"({n_gone * tau / 60:.1f} hours with tau = {tau:.0f} min)")
plt.axhline(1, color=INK, ls="--", lw=1)
plt.xlabel(r"generations $t/\tau$"); plt.ylabel("molecules per cell"); plt.ylim(0.02, 3000)
plt.legend(); plt.show()

**🔮 A question for the class.** The equation says that after 3 generations *every* cell has
exactly 125 molecules, and after 9 generations about 2. If we looked at these cells under a
microscope, would they all have the same number of molecules?

---
## 4. Partitioning at cell division

A mother cell with $N$ molecules divides. **Each molecule flips a coin**: with probability $1/2$
it goes to daughter 1, otherwise to daughter 2. The number $n_1$ in daughter 1 follows the
binomial distribution of the bootcamp, $k$ heads in $N$ tosses with probability $p$ of heads,

$$P(k) = \binom{N}{k}\, p^{k}\,(1-p)^{N-k},$$

with $k = n_1$ and $p = 1/2$, which gives $P(n_1) = \binom{N}{n_1}\,\dfrac{1}{2^N}$.

In [ ]:
def binomial_probability(N, k, p=0.5):
    '''Probability of k successes in N independent trials, each with probability p (bootcamp).
    Here: probability that daughter 1 receives k of the N molecules.'''
    return math.comb(N, k) * p**k * (1 - p)**(N - k)

### 4a. The distribution of $n_1$

**🔮 Predict:** the mother has $N = 10$ molecules. What is the probability that daughter 1 gets
exactly 5? And none? How does the distribution change for $N = 50$ and $N = 100$?

In [ ]:
print(f"N = 10: P(exactly 5) = {binomial_probability(10, 5):.3f}")
print(f"N = 10: P(none)      = {binomial_probability(10, 0):.4f}")

plt.figure(figsize=(7, 3.8))
for N, col in [(10, CORAL), (50, BLUE), (100, INK)]:
    n1 = np.arange(0, N + 1)
    P = [binomial_probability(N, int(k)) for k in n1]
    plt.plot(n1, P, "o-", color=col, ms=3, lw=1, label=f"N = {N}")
plt.xlabel("molecules in daughter 1, $n_1$"); plt.ylabel("probability"); plt.xlim(0, 80)
plt.legend(); plt.show()

### 4b. Few molecules, noisy daughters

The binomial gives $\langle n_1\rangle = N/2$ and $\sigma_{n_1} = \sqrt{N}/2$, so the **relative
noise** is $\sigma_{n_1}/\langle n_1\rangle = 1/\sqrt{N}$.

**🔮 Predict:** by how much does the relative noise decrease when $N$ is multiplied by 100?

In [ ]:
for N in [10, 100, 1000]:
    mean = N / 2
    sd = math.sqrt(N) / 2
    print(f"N = {N:4d}: mean {mean:6.1f}, standard deviation {sd:5.2f}, relative noise {sd / mean:.3f}")

N_values = np.logspace(0, 3.5, 100)
plt.figure(figsize=(6, 3.5))
plt.loglog(N_values, 1 / np.sqrt(N_values), color=INK, lw=1.5, label=r"$1/\sqrt{N}$")
plt.loglog([10, 100, 1000], [1 / math.sqrt(N) for N in [10, 100, 1000]], "o", color=BLUE, ms=8)
plt.xlabel("molecules in the mother, $N$"); plt.ylabel("relative noise"); plt.legend(); plt.show()

Back to Section 3: when the protein is down to about 10 molecules per cell, sister cells differ by
about 30% of the mean, just because of the coin flips at division.

At the board we then derived what an experiment can measure: with $I = \nu\, n$ the fluorescence
intensity of a cell,

$$\langle (I_1 - I_2)^2 \rangle = \nu\, I_\mathrm{tot},$$

so the brightness $\nu$ of a single molecule is the slope of $\langle (I_1 - I_2)^2 \rangle$
against $I_\mathrm{tot}$.

---
### What we did

| question | model | tool | answer |
|---|---|---|---|
| how fast does a gene respond? | $dY/dt = \beta - \alpha Y$ | pen and paper | $T_{1/2} = \ln 2/\alpha$, independent of $\beta$ |
| when is a protein gone? | $dY/dt = -\alpha Y$ | pen and paper | halved every generation; < 1 molecule after $\log_2 Y_\mathrm{st}$ generations |
| why do sister cells differ? | one coin per molecule | binomial distribution | relative noise $1/\sqrt{N}$ |
| how bright is one molecule? | $I = \nu n$ | derivation | $\langle (I_1-I_2)^2\rangle = \nu I_\mathrm{tot}$ |